<a href="https://colab.research.google.com/github/pauljit/Military_service_coding/blob/main/Graphic_Engine/Turtle_week2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 8일차(26.02.20) (4x4 행렬 클래스 구현, 스케일 조절, 벡터 적용, 이동, 로테이션)

%%writefile mymath.py

import math

class Vector2:
  def __init__(self, x, y):
    self.x = x
    self.y = y

  #벡터끼리 더하기
  def add(self, other_vector):
    return Vector2(self.x + other_vector.x, self.y + other_vector.y)

  #벡터끼리 빼기
  def minus(self, other_vector):
    return Vector2(self.x - other_vector.x, self.y - other_vector.y)

  #벡터의 곱셈
  def multiply(self, factor):
    return Vector2(self.x *factor, self.y * factor)

  #벡터의 길이
  def magnitude(self):
    return math.sqrt((self.x ** 2) + (self.y ** 2))

  #벡터 정보 출력
  def status(self):
    print(f"x좌표: {self.x}, y좌표: {self.y}, 길이: {self.magnitude()}")

  #벡터 정규화 (방향을 알기 위한 용도)
  def normalize(self):
    mag = self.magnitude()
    if mag == 0:
      return Vector2(0,0)
    else:
      return Vector2(self.x/mag, self.y/mag)

  #벡터 내적 (+-0의 상태에 따라 방향의 일치성 확인, 양에 따라 빛의 반사율 확인)
  def dot(self, other_vector):
    return ((self.x*other_vector.x) + (self.y * other_vector.y))


class Vector3:
  # 1. 생성자 (z축 추가)
  def __init__(self, x=0, y=0, z=0):
    self.x = x
    self.y = y
    self.z = z

  # 2. 덧셈 (z축 끼리도 더해주세요)
  def add(self, other):
    return Vector3(self.x + other.x, self.y + other.y, self.z + other.z)

  # 3. 뺄셈
  def minus(self, other):
    return Vector3(self.x - other.x, self.y - other.y, self.z - other.z)

  # 4. 스칼라 곱셈
  def multiply(self, scalar):
    return Vector3(self.x * scalar, self.y * scalar, self.z * scalar)

  # 5. 길이 구하기
  def magnitude(self):
    return math.sqrt(self.x ** 2 + self.y ** 2 + self.z **2)

  # 6. 정규화 (방향 벡터로 만들기)
  def normalize(self):
    mag = self.magnitude()
    if mag == 0:
      return Vector3(0, 0, 0)
    else:
      return Vector3(self.x / mag, self.y / mag, self.z / mag)

  # 7. 내적 (조명 계산의 핵심)
  def dot(self, other):
    return (self.x * other.x) + (self.y * other.y) + (self.z * other.z)

  # 8. 벡터 좌표 출력 (:.2f는 소수점 2자리까지만 출력한다는 뜻)
  def status(self):
    print(f"Vector3(x: {self.x:.2f}, y: {self.y:.2f}, z: {self.z:.2f})")

  #  9. 위치 벡터끼리의 거리 (서로 뺀 값의 길이)
  def distance(self, other):
    return self.minus(other).magnitude()

  #  10. 벡터끼리의 각도 (서로 정규화한 벡터의 내적의 아크코사인)(적이 보는 각도 확인 가능)
  def angle(self, other):
     dot_product = self.normalize().dot(other.normalize())
     # 1.0001등의 값을 내어 acos가 오류를 내지 않기 위해 -1.0 ~ 1.0으로 바꾸기
     dot_product = max(-1.0, min(1.0, dot_product))
     radian = math.acos(dot_product)
     return math.degrees(radian)

  # 11. 외적 (두 벡터의 수직인 법선 벡터)(삼각 폴리곤의 수직을 구하여 빛 반사 및 culling 최적화)
  def cross(self, other):
    normal_vector = Vector3()
    normal_vector.x = self.y * other.z - self.z * other.y
    normal_vector.y = self.x * other.z - self.z * other.x
    normal_vector.z = self.x * other.y - self.y * other.x
    return normal_vector


# 3D 공간에서 TRS(Translation, Rotation, Scale)을 조정하는 4*4 매트릭스
class Matrix4:
  # 1. 생성자 (단위 행렬(identity matrix))
  def __init__(self):
    self.matrix = [
        [1.0, 0.0, 0.0, 0.0],
        [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0],
        [0.0, 0.0, 0.0, 1.0]
    ]


  # 2. 행렬 출력
  def status(self):
    for row in self.matrix:
      print(f"{row[0]:.2f},{row[1]:.2f},{row[2]:.2f},{row[3]:.2f}")
    print("--"*10)


  # 3. 스케일 조절 (첫 세 행의 각 x, y, z 변형)
  def scale(self,scale_x,scale_y,scale_z):
    self.matrix[0][0] = scale_x
    self.matrix[1][1] = scale_y
    self.matrix[2][2] = scale_z


  # 4. 행렬을 벡터에 적용 (위치 벡터를 )
  def mul_vector(self, vector):
    # w는 3차원 벡터를 4*4 행렬에서 구현하기 위해 가상으로 만든 개념, 벡터에서 이동값을 담당
    w = 1.0
    new_x = (self.matrix[0][0] * vector.x) + (self.matrix[0][1] * vector.y) + (self.matrix[0][2] * vector.z) + (self.matrix[0][3] * w)
    new_y = (self.matrix[1][0] * vector.x) + (self.matrix[1][1] * vector.y) + (self.matrix[1][2] * vector.z) + (self.matrix[1][3] * w)
    new_z = (self.matrix[2][0] * vector.x) + (self.matrix[2][1] * vector.y) + (self.matrix[2][2] * vector.z) + (self.matrix[2][3] * w)
    return Vector3(new_x, new_y, new_z)


  # 5. 이동 행렬 (mul_vector의 w의 값을 이용하여 위치 벡터를 이동)
  def translate(self, translate_x, translate_y, translate_z):
    # 이러면 기존의 0이었던 4번째 열의 값이 변형되어 x,y,z값이 바뀜
    self.matrix[0][3] = translate_x
    self.matrix[1][3] = translate_y
    self.matrix[2][3] = translate_z


  # 6. z축 회전 행렬 (x, y 행만 변형)
  def rotate_z(self, degree):
    rad = math.radians(degree)
    cos = math.cos(rad)
    sin = math.sin(rad)
    self.matrix[0][0] = cos
    self.matrix[0][1] = -sin
    self.matrix[1][0] = sin
    self.matrix[1][1] = cos


  #7. y축 회전 행렬 (x, z 행만 변형)
  def rotate_y(self, degree):
    rad = math.radians(degree)
    cos = math.cos(rad)
    sin = math.sin(rad)
    self.matrix[0][0] = cos
    self.matrix[0][2] = sin
    self.matrix[2][0] = -sin
    self.matrix[2][2] = cos


  #8. x축 회전 행렬 (y, z 행만 변형)
  def rotate_x(self, degree):
    rad = math.radians(degree)
    cos = math.cos(rad)
    sin = math.sin(rad)
    self.matrix[1][1] = cos
    self.matrix[1][2] = -sin
    self.matrix[2][1] = sin
    self.matrix[2][2] = cos

Overwriting mymath.py


In [ ]:
# 8일차(26.02.20) (4x4 행렬 클래스 구현, 스케일 조절)

from mymath import *

#기본 행렬 출력
new_mat = Matrix4()
new_mat.status()

#스케일 조절된(y축 2배) 벡터 출력
new_mat.scale(1.0,2.0,1.0)
new_mat.status()

#행렬에 영향받아 새로운 좌표를 가진 벡터 출력
mat_vec = new_mat.mul_vector(Vector3(1.0,2.0,3.0))
mat_vec.status()

# 이동 행렬과 위치 벡터
position_vector = Vector3(1, 3, -5)
print("현재 좌표: ")
position_vector.status()
translate_matrix = Matrix4()
translate_matrix.translate(10.0, -2.0, 4.0)
new_position_vector = translate_matrix.mul_vector(position_vector)
print("이동된 좌표: ")
new_position_vector.status()

#z축, y축, x축 회전
z_matrix = Matrix4()
y_matrix = Matrix4()
x_matrix = Matrix4()

z_matrix.rotate_z(90)
y_matrix.rotate_y(90)
x_matrix.rotate_x(90)

r_vector = Vector3(10, 5, 9)
print("현재 좌표:")
r_vector.status()

z_vector = z_matrix.mul_vector(r_vector)
y_vector = y_matrix.mul_vector(r_vector)
x_vector = x_matrix.mul_vector(r_vector)

print("z축으로 회전된 좌표:")
z_vector.status()
print("y축으로 회전된 좌표:")
y_vector.status()
print("x축으로 회전된 좌표:")
x_vector.status()


1.00,0.00,0.00,0.00
0.00,1.00,0.00,0.00
0.00,0.00,1.00,0.00
0.00,0.00,0.00,1.00
--------------------
1.00,0.00,0.00,0.00
0.00,2.00,0.00,0.00
0.00,0.00,1.00,0.00
0.00,0.00,0.00,1.00
--------------------
Vector3(x: 1.00, y: 4.00, z: 3.00)
현재 좌표: 
Vector3(x: 1.00, y: 3.00, z: -5.00)
이동된 좌표: 
Vector3(x: 11.00, y: 1.00, z: -1.00)
현재 좌표:
Vector3(x: 10.00, y: 5.00, z: 9.00)
z축으로 회전된 좌표:
Vector3(x: -5.00, y: 10.00, z: 9.00)
y축으로 회전된 좌표:
Vector3(x: 9.00, y: 5.00, z: -10.00)
x축으로 회전된 좌표:
Vector3(x: 10.00, y: -9.00, z: 5.00)
